# Lab 03 solution, notebook 1: add a second tool, and say why the run ended

The worked answers to the two challenge tasks of `03a_meet_the_agent_loop.ipynb`. The setup repeats what the tasks
need from the notebook, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
from openai import OpenAI, omit

client = OpenAI(timeout=60)

SYSTEM = (
    "You are playing an envelope hunt. A prize is hidden in one of the numbered envelopes. "
    "Open envelopes with open_envelope and follow the clues. Answer in one sentence."
)
QUESTION = "Where is the prize? Start with envelope 1."

## Task 1: Add a second tool

The new hunt goes through a drawer: envelope 1 points to the kitchen drawer, the kitchen drawer points to envelope
3, and envelope 3 holds the prize. The drawers are a second dictionary, and `open_drawer` is a second plain
function with its own `TOOL CALLED` print.

In [3]:
ENVELOPES = {
    1: "Clue: look in the kitchen drawer.",
    2: "This envelope is empty.",
    3: "The prize! Two tickets to the cinema.",
    4: "This envelope is empty.",
    5: "This envelope is empty.",
}

DRAWERS = {
    "kitchen": "Clue: look in envelope 3.",
    "desk": "This drawer is empty.",
}


def open_envelope(number: int) -> str:
    """Open one envelope and read what is inside."""
    print(f"TOOL CALLED: open_envelope({number})")
    return ENVELOPES.get(number, "There is no envelope with that number.")


def open_drawer(room: str) -> str:
    """Open the drawer in one room and read what is inside."""
    print(f"TOOL CALLED: open_drawer({room})")
    return DRAWERS.get(room, "There is no drawer in that room.")

Each tool gets its own schema. The second one takes a string, `room`.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "open_envelope",
            "description": "Open one envelope and read what is inside.",
            "parameters": {
                "type": "object",
                "properties": {"number": {"type": "integer", "description": "The envelope's number"}},
                "required": ["number"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "open_drawer",
            "description": "Open the drawer in one room and read what is inside.",
            "parameters": {
                "type": "object",
                "properties": {"room": {"type": "string", "description": "The room, for example 'kitchen'"}},
                "required": ["room"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
]

With two tools the loop can no longer send every call to `open_envelope`. `FUNCTIONS` maps each tool name to its
function, and the loop looks up `call.function.name` in it. That lookup is the one line of the loop that changes.

In [5]:
FUNCTIONS = {"open_envelope": open_envelope, "open_drawer": open_drawer}


def run_loop(question, max_turns=8):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        messages.append(reply)

        if response.choices[0].finish_reason != "tool_calls":
            return reply.content

        for call in reply.tool_calls:
            function = FUNCTIONS[call.function.name]
            result = function(**json.loads(call.function.arguments))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return None


print(run_loop(QUESTION))

TOOL CALLED: open_envelope(1)


TOOL CALLED: open_drawer(kitchen)


TOOL CALLED: open_envelope(3)


The prize is in envelope 3: two tickets to the cinema.


Three `TOOL CALLED` lines, in the order of the clues: `open_envelope(1)`, `open_drawer(kitchen)`, `open_envelope(3)`.
The model moved from one tool to the other because the clue said so, and the dictionary sent each call to the right
function. The answer names envelope 3.

## Task 2: Say why the run ended

The loop has two exits, and each `return` now gives two values: the answer and the reason. Python returns them as a
pair, which the caller unpacks into two names.

The hunt is the one with no end, where every envelope points to the next number. `FUNCTIONS` holds the function it
was given, not the name, so after defining `open_envelope` again we put the new function in the dictionary.

In [6]:
def run_loop_with_reason(question, max_turns=8):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        messages.append(reply)

        if response.choices[0].finish_reason != "tool_calls":
            return reply.content, "answer"

        for call in reply.tool_calls:
            function = FUNCTIONS[call.function.name]
            result = function(**json.loads(call.function.arguments))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return None, "turn cap"


def open_envelope(number: int) -> str:
    """Open one envelope and read what is inside."""
    print(f"TOOL CALLED: open_envelope({number})")
    return f"Clue: look in envelope {number + 1}."


FUNCTIONS["open_envelope"] = open_envelope

answer, reason = run_loop_with_reason(QUESTION, max_turns=5)
print("answer:", answer)
print("reason:", reason)

TOOL CALLED: open_envelope(1)


TOOL CALLED: open_envelope(2)


TOOL CALLED: open_envelope(3)


TOOL CALLED: open_envelope(4)


TOOL CALLED: open_envelope(5)
answer: None
reason: turn cap


Five `TOOL CALLED` lines, for envelopes 1 to 5, then `answer: None` and `reason: turn cap`. The caller no longer has
to work out from `None` what happened: it can tell the person who asked that the run was stopped before an answer
was found.